In [0]:
# =============================================================
# CELDA 1: SETUP — Imports, Credenciales, Config S3
# =============================================================
import json
import boto3
import unicodedata
import re
import os
import sys
import importlib
from functools import reduce
from pyspark.sql.functions import (
    col, coalesce, regexp_extract, trim, lit,
    when, length, lower, row_number, desc, current_timestamp,
    regexp_replace, initcap
)
from pyspark.sql.types import StringType
from pyspark.sql.window import Window
from pyspark.sql import functions as F

# Credenciales: Databricks secrets (producción) → archivo local (desarrollo)
try:
    config = {
        "aws_access_key": dbutils.secrets.get(scope="aws", key="access_key"),
        "aws_secret_key": dbutils.secrets.get(scope="aws", key="secret_key"),
    }
    print("✅ Credenciales desde Databricks Secrets.")
except Exception:
    try:
        with open("aws_secrets.json", "r") as f:
            config = json.load(f)
        print("✅ Credenciales desde aws_secrets.json (local).")
    except FileNotFoundError:
        raise SystemExit("❌ Credenciales no disponibles. Configura Databricks Secrets (scope='aws') o coloca aws_secrets.json.")
    except json.JSONDecodeError:
        raise SystemExit("❌ aws_secrets.json inválido.")

AWS_KEY = config["aws_access_key"]
AWS_SECRET = config["aws_secret_key"]
BUCKET = "bronce-scrap-date"

S3_OPTS = {
    "fs.s3a.access.key": AWS_KEY,
    "fs.s3a.secret.key": AWS_SECRET,
    "fs.s3a.endpoint": "s3.amazonaws.com",
}

# Reglas por fuente derivadas del análisis local de raw parquet
_candidates = []
try:
    _nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    _repo_dir = "/Workspace" + str(_nb_path).rsplit("/", 2)[0]
    _candidates.append(_repo_dir)
except Exception:
    pass
_vsc_file = globals().get("__vsc_ipynb_file__", "")
if _vsc_file:
    _candidates.append(os.path.dirname(os.path.abspath(_vsc_file)))
_candidates.append(os.getcwd())
for _candidate in _candidates:
    if _candidate and _candidate not in sys.path:
        sys.path.insert(0, _candidate)

try:
    import source_quality_rules
    importlib.reload(source_quality_rules)
    from source_quality_rules import SOURCE_PRICE_RULES, SOURCE_REQUIRED_SIGNALS
    print("✅ source_quality_rules.py cargado.")
except Exception:
    SOURCE_PRICE_RULES = {}
    SOURCE_REQUIRED_SIGNALS = {}
    print("⚠️ source_quality_rules.py no disponible; usando reglas por defecto.")

# ─────────────────────────────────────────────────────────────
# UDFs DE NORMALIZACIÓN (se aplican en Silver, antes de todo)
# ─────────────────────────────────────────────────────────────

def _normalizar_tipo(tipo_raw):
    """396 variantes libres → 6 categorías canónicas."""
    if tipo_raw is None:
        return "otro"
    t = tipo_raw.lower().strip()
    if any(x in t for x in ["apto", "apart", "piso", "estudio", "loft", "penthouse", "duplex"]):
        return "apartamento"
    if any(x in t for x in ["casa", "chalet", "villa", "finca", "cabana", "cabañ"]):
        return "casa"
    if any(x in t for x in ["oficin", "consultori"]):
        return "oficina"
    if any(x in t for x in ["local", "bodega", "comerci", "nave"]):
        return "local_comercial"
    if any(x in t for x in ["lote", "terreno", "parcela"]):
        return "lote"
    return "otro"

normalizar_tipo_udf = F.udf(_normalizar_tipo, StringType())


def _normalizar_estado(estado_raw):
    """Variantes de estado → 4 categorías."""
    if estado_raw is None:
        return "desconocido"
    t = estado_raw.lower().strip()
    if any(x in t for x in ["nuevo", "new", "estrenar"]):
        return "nuevo"
    if any(x in t for x in ["usado", "second", "segunda"]):
        return "usado"
    if any(x in t for x in ["proyecto", "plano", "sobre plano", "preventa"]):
        return "sobre_planos"
    if any(x in t for x in ["remodelado", "renovado"]):
        return "remodelado"
    return "desconocido"

normalizar_estado_udf = F.udf(_normalizar_estado, StringType())


def _normalizar_ubicacion(text):
    """Normaliza ubicación: quita acentos, lowercase, limpia separadores."""
    if text is None:
        return None
    nfkd = unicodedata.normalize("NFKD", text)
    text = "".join(c for c in nfkd if not unicodedata.combining(c))
    text = text.lower().strip()
    for noise in ["d.c.", "d.c", "dc", "colombia", "departamento", "municipio"]:
        text = text.replace(noise, "")
    text = re.sub(r"[,\-–—/|·•()]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

normalizar_ubicacion_udf = F.udf(_normalizar_ubicacion, StringType())


_CIUDADES_CONOCIDAS = {
    "bogota", "medellin", "cali", "barranquilla", "cartagena",
    "bucaramanga", "pereira", "manizales", "cucuta", "ibague",
    "santa marta", "villavicencio", "pasto", "monteria", "neiva",
    "armenia", "popayan", "valledupar", "sincelejo", "tunja",
    "envigado", "sabaneta", "itagui", "bello", "soacha",
    "chia", "zipaquira", "cajica", "cota", "funza",
    "mosquera", "tocancipa", "la calera", "sopo",
    "rionegro", "girardot", "floridablanca", "piedecuesta",
}

def _extraer_ciudad(ubicacion_norm):
    if ubicacion_norm is None:
        return "otra_ciudad"
    for city in _CIUDADES_CONOCIDAS:
        if city in ubicacion_norm:
            return city
    return "otra_ciudad"

extraer_ciudad_udf = F.udf(_extraer_ciudad, StringType())

In [0]:
# =============================================================
# CELDA 2: FUNCIÓN — Normalización Universal Multi-Fuente
# =============================================================
# Cada portal usa nombres diferentes para el mismo concepto:
#   precio/price, ubicacion/location/city, titulo/title, url/listing_url
# Esta función usa coalesce() para mapear TODAS las variantes sin if/else.
# Además aplica normalización estricta de tipo_inmueble, estado, ubicación.

def _safe_extract_number(col_expr, pattern, cast_type="double"):
    """regexp_extract retorna '' cuando no hay match → cast falla.
    Esta helper retorna NULL en vez de '' antes del cast."""
    extracted = regexp_extract(col_expr, pattern, 1)
    return when(length(extracted) > 0, extracted.cast(cast_type))

def normalizar_fuente(fuente):
    """Lee Bronze Delta de una fuente y normaliza a schema Silver unificado."""
    ruta = f"s3a://{BUCKET}/bronze/{fuente}/"
    print(f"📖 {fuente}...")

    try:
        reader = spark.read.format("delta")
        for k, v in S3_OPTS.items():
            reader = reader.option(k, v)
        df = reader.load(ruta)
    except Exception as e:
        print(f"  ⚠️ Saltando {fuente}: {str(e)[:120]}")
        return None

    cols = set(df.columns)
    def safe(name):
        """Retorna col(name) si existe, lit(None) si no — evita errores por schema variable."""
        return col(name) if name in cols else lit(None).cast("string")

    required = SOURCE_REQUIRED_SIGNALS.get(fuente, [])
    missing_required = [c for c in required if c not in cols]
    if missing_required:
        print(f"  ⚠️ {fuente}: faltan señales esperadas: {missing_required}")

    price_rules = SOURCE_PRICE_RULES.get(
        fuente,
        {"min_price": 20_000_000, "max_price": 20_000_000_000},
    )
    min_price = price_rules["min_price"]
    max_price = price_rules["max_price"]

    # ═══════════════════════════════════════════════════════════
    # PASO 1: MAPEO UNIVERSAL (Bronze → schema Silver crudo)
    # ═══════════════════════════════════════════════════════════
    df_norm = df.select(
        col("id_inmueble").alias("id_original"),

        coalesce(
            safe("fecha_extraccion"), safe("extracted_at"),
            safe("ingestion_timestamp"), safe("bronze_ingested_at")
        ).cast("timestamp").alias("fecha_extraccion"),

        coalesce(safe("source_system"), safe("portal"), safe("source")).alias("fuente"),

        coalesce(safe("url"), safe("listing_url")).alias("url"),

        coalesce(safe("titulo"), safe("title")).alias("titulo"),

        coalesce(safe("ubicacion"), safe("location"), safe("city")).alias("ubicacion_raw"),

        coalesce(
            safe("precio_num").cast("long"),
            _safe_extract_number(safe("price"), r"(\d{5,})", "long")
        ).alias("precio_num"),

        coalesce(
            safe("area_m2").cast("double"),
            _safe_extract_number(safe("area"), r"(\d+\.?\d*)", "double")
        ).alias("area_m2"),

        _safe_extract_number(
            coalesce(safe("habitaciones"), safe("rooms"), safe("bedrooms")), r"(\d+)", "int"
        ).alias("habitaciones"),

        _safe_extract_number(coalesce(safe("banos"), safe("bathrooms"), safe("baths")), r"(\d+)", "int").alias("banos"),

        _safe_extract_number(coalesce(safe("garajes"), safe("parking"), safe("garages")), r"(\d+)", "int").alias("garajes"),

        coalesce(safe("tipo_inmueble"), safe("property_type"), safe("category")).alias("tipo_inmueble_raw"),

        coalesce(safe("estado_inmueble"), safe("status"), safe("condition")).alias("estado_inmueble_raw"),

        safe("source_file").alias("source_file"),
        safe("batch_id").alias("batch_id"),
    )

    # ═══════════════════════════════════════════════════════════
    # PASO 2: FILTROS DE CALIDAD (gates estrictos)
    # ═══════════════════════════════════════════════════════════
    n_raw = df_norm.count()
    n_with_price = df_norm.filter(col("precio_num").isNotNull()).count()

    df_clean = df_norm.filter(
        (col("precio_num").isNotNull()) &
        (col("precio_num") > min_price) &
        (col("precio_num") < max_price) &
        (col("id_original").isNotNull())
    )

    n_clean_price = df_clean.count()
    print(
        f"  🧪 {fuente}: precio válido {n_clean_price}/{n_raw} "
        f"(con precio parseado {n_with_price}/{n_raw}, gate {min_price:,} - {max_price:,})"
    )

    # ═══════════════════════════════════════════════════════════
    # PASO 3: NORMALIZACIÓN DE CAMPOS (la clave del fix)
    # ═══════════════════════════════════════════════════════════
    df_clean = (
        df_clean
        .withColumn("tipo_inmueble", normalizar_tipo_udf(col("tipo_inmueble_raw")))
        .drop("tipo_inmueble_raw")
        .withColumn("estado_inmueble", normalizar_estado_udf(col("estado_inmueble_raw")))
        .drop("estado_inmueble_raw")
        .withColumn(
            "ubicacion_raw",
            when(
                col("ubicacion_raw").isNull() |
                (length(trim(col("ubicacion_raw"))) < 3) |
                (lower(col("ubicacion_raw")) == "nan"),
                lit("Sin Ubicación")
            ).otherwise(trim(col("ubicacion_raw")))
        )
        .withColumn("ubicacion_norm", normalizar_ubicacion_udf(col("ubicacion_raw")))
        .withColumn("city_token", extraer_ciudad_udf(col("ubicacion_norm")))
        .withColumn(
            "titulo",
            when(col("titulo").isNull(), lit(None))
            .otherwise(regexp_replace(trim(col("titulo")), r"\s+", " "))
        )
        .withColumn(
            "area_m2",
            when((col("area_m2") >= 10) & (col("area_m2") <= 2000), col("area_m2"))
            .otherwise(lit(None))
        )
        .withColumn(
            "habitaciones",
            when((col("habitaciones") >= 0) & (col("habitaciones") <= 20), col("habitaciones"))
            .otherwise(lit(None))
        )
        .withColumn(
            "banos",
            when((col("banos") >= 0) & (col("banos") <= 15), col("banos"))
            .otherwise(lit(None))
        )
        .withColumn(
            "garajes",
            when((col("garajes") >= 0) & (col("garajes") <= 10), col("garajes"))
            .otherwise(lit(None))
        )
    )

    n_clean = df_clean.count()
    print(f"  ✅ {fuente}: {n_clean} válidos (de {n_raw} raw, {n_raw - n_clean} descartados)")
    return df_clean

In [0]:
# =============================================================
# CELDA 3: EJECUCIÓN — Descubrimiento Dinámico + Unión + Dedup
# =============================================================

# Descubrir fuentes dinámicamente desde Bronze (NO hardcoded)
s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_KEY,
    aws_secret_access_key=AWS_SECRET,
    region_name="us-east-1",
)

response = s3.list_objects_v2(Bucket=BUCKET, Prefix="bronze/", Delimiter="/")
fuentes = [
    p["Prefix"].replace("bronze/", "").strip("/")
    for p in response.get("CommonPrefixes", [])
    if p["Prefix"].strip("/") != "bronze"
]
print(f"🔍 Fuentes en Bronze: {fuentes}")

# Normalizar cada fuente
dfs = []
for f in fuentes:
    df = normalizar_fuente(f)
    if df is not None:
        dfs.append(df)

if not dfs:
    raise SystemExit("❌ No hay datos para procesar en Bronze.")

# Unión (allowMissingColumns maneja columnas que no existen en todas las fuentes)
df_union = reduce(lambda a, b: a.unionByName(b, allowMissingColumns=True), dfs)
print(f"\n📊 Total unificado: {df_union.count()} registros")

# Deduplicación: conservar el registro más reciente por (id_original, fuente)
w = Window.partitionBy("id_original", "fuente").orderBy(desc("fecha_extraccion"))

df_silver = (
    df_union
    .withColumn("_rank", row_number().over(w))
    .filter(col("_rank") == 1)
    .drop("_rank")
    .withColumn("silver_processed_at", current_timestamp())
)

n_final = df_silver.count()
print(f"📉 Registros únicos: {n_final}")

# ═══════════════════════════════════════════════════════════════
# DIAGNÓSTICO DE CALIDAD — verificar que las normalizaciones funcionan
# ═══════════════════════════════════════════════════════════════
print("\n🔬 DIAGNÓSTICO POST-NORMALIZACIÓN:")

# tipo_inmueble: debe ser ~6 categorías
print("\n── tipo_inmueble ──")
df_silver.groupBy("tipo_inmueble").count().orderBy(desc("count")).show(truncate=False)

# estado_inmueble: debe ser ~4 categorías
print("── estado_inmueble ──")
df_silver.groupBy("estado_inmueble").count().orderBy(desc("count")).show(truncate=False)

# city_token
print("── city_token (top 20) ──")
df_silver.groupBy("city_token").count().orderBy(desc("count")).show(20, truncate=False)
n_cities = df_silver.select("city_token").distinct().count()
print(f"   Total ciudades únicas: {n_cities}")

# Nulls por columna
print("── Completitud ──")
for c in ["precio_num", "area_m2", "habitaciones", "banos", "garajes", "tipo_inmueble", "ubicacion_raw", "city_token"]:
    null_pct = df_silver.filter(col(c).isNull()).count() / n_final * 100
    print(f"   {c}: {100 - null_pct:.1f}% completo ({null_pct:.1f}% nulls)")

# Precio por portal
print("\n── Registros por portal ──")
df_silver.groupBy("fuente").count().orderBy(desc("count")).show(truncate=False)

In [0]:
# =============================================================
# CELDA 4: GUARDAR — Silver (Delta) + Gold Consumable (Parquet)
# =============================================================

# Silver: tabla maestra de-duplicada y normalizada
ruta_silver = f"s3a://{BUCKET}/silver/master_inmuebles/"
writer = df_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
for k, v in S3_OPTS.items():
    writer = writer.option(k, v)
writer.save(ruta_silver)
print(f"🥈 Silver guardado: {ruta_silver}")
print(f"   Columnas: {df_silver.columns}")

# Gold consumable: snapshot para API (un solo archivo parquet)
ruta_gold = f"s3a://{BUCKET}/gold/app_inmuebles/"
writer = df_silver.coalesce(1).write.format("parquet").mode("overwrite")
for k, v in S3_OPTS.items():
    writer = writer.option(k, v)
writer.save(ruta_gold)
print(f"🥇 Gold consumable: {ruta_gold}")